# Module 1 Homework - 2026 Cohort

This notebook contains reproducible solutions for Questions 1-4 and writing prompts for the optional Questions 5-6. Run the cells from top to bottom because the results depend on current web and market data.

In [2]:
%pip install -q yfinance pandas requests lxml beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\atimokhin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from io import StringIO

import numpy as np
import pandas as pd
import requests
import yfinance as yf

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## Question 1: S&P 500 additions

Find the full year since 2020 with the most additions to the current S&P 500 constituents. The additional result counts current constituents that joined more than 20 years ago.

In [4]:
sp500_url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}
response = requests.get(sp500_url, headers=headers, timeout=30)
response.raise_for_status()
sp500 = pd.read_html(StringIO(response.text), match="Date added")[0]

sp500_companies = sp500[["Symbol", "Security", "Date added"]].copy()
sp500_companies["Date added"] = pd.to_datetime(
    sp500_companies["Date added"], errors="coerce"
)
sp500_companies["Year added"] = sp500_companies["Date added"].dt.year.astype("Int64")
sp500_companies.head()

,Symbol,Security,Date added,Year added
0,MMM,3M,1957-03-04,1957
1,AOS,A. O. Smith,2017-07-26,2017
2,ABT,Abbott Laboratories,1957-03-04,1957
3,ABBV,AbbVie,2012-12-31,2012
4,ACN,Accenture,2011-07-06,2011


In [5]:
additions_by_year = (
    sp500_companies.loc[sp500_companies["Year added"] >= 2020, "Year added"]
    .value_counts()
    .sort_index()
)
latest_full_year = pd.Timestamp.today().year - 1
full_year_additions = additions_by_year[additions_by_year.index <= latest_full_year]
highest_addition_year = int(full_year_additions.idxmax())
highest_addition_count = int(full_year_additions.max())

twenty_year_cutoff = pd.Timestamp.today().normalize() - pd.DateOffset(years=20)
more_than_20_years = int(
    sp500_companies["Date added"].lt(twenty_year_cutoff).sum()
)

display(additions_by_year.rename("additions").to_frame())
print(f"Answer: {highest_addition_year} had the most additions ({highest_addition_count}).")
print(f"Additional: {more_than_20_years} current constituents have been in the index for more than 20 years.")

,additions
Year added,
2020,10
2021,10
2022,15
2023,15
2024,16
2025,18
2026,13


,additions
Year added,
2020,10
2021,10
2022,15
2023,15
2024,16
2025,18
2026,13


Answer: 2025 had the most additions (18).
Additional: 224 current constituents have been in the index for more than 20 years.


## Question 2: World indexes YTD through 21 August 2026

Yahoo Finance treats `end` as exclusive, so the download ends on 22 August to include the 21 August close. The comparison set contains ten non-US indexes; the S&P 500 is the benchmark. Currency effects are ignored.

In [6]:
index_tickers = {
    "United States - S&P 500": "^GSPC",
    "China - Shanghai Composite": "000001.SS",
    "Hong Kong - Hang Seng": "^HSI",
    "Australia - S&P/ASX 200": "^AXJO",
    "India - Nifty 50": "^NSEI",
    "Canada - S&P/TSX Composite": "^GSPTSE",
    "Germany - DAX": "^GDAXI",
    "United Kingdom - FTSE 100": "^FTSE",
    "Japan - Nikkei 225": "^N225",
    "Mexico - IPC": "^MXX",
    "Brazil - Ibovespa": "^BVSP",
}

index_prices = yf.download(
    list(index_tickers.values()),
    start="2026-01-01",
    end="2026-08-22",
    auto_adjust=False,
    progress=False,
)["Close"]
index_prices = index_prices.rename(columns={ticker: name for name, ticker in index_tickers.items()})
index_prices.tail()

Ticker,China - Shanghai Composite,Australia - S&P/ASX 200,Brazil - Ibovespa,United Kingdom - FTSE 100,Germany - DAX,United States - S&P 500,Canada - S&P/TSX Composite,Hong Kong - Hang Seng,Mexico - IPC,Japan - Nikkei 225,India - Nifty 50
Date,,,,,,,,,,,
2026-08-17,"3,982.6541","9,073.2002","166,784.0000","10,720.2998","26,338.6094","7,745.0601","36,667.8984","25,453.2305","64,254.9805","69,220.2500","24,287.6504"
2026-08-18,"3,990.3040","9,070.0000","166,335.0000","10,728.0000","26,128.3594","7,691.7598","36,367.8984","25,471.1504","63,933.6914","67,460.7266","24,154.9004"
2026-08-19,"3,894.4221","9,053.7998","167,830.0000","10,743.4004","26,091.3301","7,707.9800","36,401.8008","25,495.0703","63,999.2617","65,326.4219","24,078.3008"
2026-08-20,"3,903.7209","9,083.7998","167,927.0000","10,748.2002","25,983.0391","7,641.1602","36,365.3984","25,698.4902","64,349.8008","66,216.7891","24,231.8496"
2026-08-21,"3,905.2029","9,058.9004","171,032.0000","10,816.5996","26,136.5605","7,674.3701","36,620.1992","26,009.4609","65,729.1797","66,016.3594","24,252.0000"


In [7]:
ytd_returns = index_prices.apply(lambda series: series.dropna().iloc[-1] / series.dropna().iloc[0] - 1)
ytd_results = ytd_returns.rename("YTD return").sort_values(ascending=False).to_frame()
sp500_ytd = ytd_returns["United States - S&P 500"]
better_than_sp500 = int((ytd_returns.drop("United States - S&P 500") > sp500_ytd).sum())

display(ytd_results.style.format({"YTD return": "{:.2%}"}))
print(f"Answer: {better_than_sp500} of the 10 non-US indexes outperformed the S&P 500.")

,YTD return
Ticker,
Japan - Nikkei 225,27.36%
Canada - S&P/TSX Composite,14.86%
United States - S&P 500,11.90%
United Kingdom - FTSE 100,8.70%
Brazil - Ibovespa,6.54%
Germany - DAX,6.51%
Australia - S&P/ASX 200,3.79%
Mexico - IPC,2.48%
Hong Kong - Hang Seng,-1.25%


Answer: 2 of the 10 non-US indexes outperformed the S&P 500.


In [8]:
comparison_end = "2026-08-22"
period_starts = {"3 years": "2023-08-21", "5 years": "2021-08-21", "10 years": "2016-08-21"}
long_prices = yf.download(
    list(index_tickers.values()),
    start=min(period_starts.values()),
    end=comparison_end,
    auto_adjust=False,
    progress=False,
)["Close"].rename(columns={ticker: name for name, ticker in index_tickers.items()})

period_comparisons = {}
for period, period_start in period_starts.items():
    period_prices = long_prices.loc[period_start:]
    returns = period_prices.apply(lambda series: series.dropna().iloc[-1] / series.dropna().iloc[0] - 1)
    benchmark_return = returns["United States - S&P 500"]
    period_comparisons[period] = {
        "S&P 500 return": benchmark_return,
        "Non-US indexes outperforming": int(
            (returns.drop("United States - S&P 500") > benchmark_return).sum()
        ),
    }

period_comparison_df = pd.DataFrame(period_comparisons).T
period_comparison_df.style.format({"S&P 500 return": "{:.2%}"})

,S&P 500 return,Non-US indexes outperforming
3 years,74.43%,2.000000
5 years,71.32%,2.000000
10 years,251.61%,1.000000


## Question 3: S&P 500 corrections

A correction episode starts at a record closing high and ends immediately before the next record high. Its trough is the minimum close in that interval. Duration is measured from the peak date to the trough date in calendar days, matching the dates in the homework examples.

In [9]:
sp500_history = yf.download(
    "^GSPC", start="1950-01-01", auto_adjust=False, progress=False
)
sp500_close = sp500_history["Close"].squeeze().dropna().rename("Close")

previous_running_high = sp500_close.cummax().shift(1)
record_high_mask = previous_running_high.isna() | sp500_close.gt(previous_running_high)
record_high_positions = np.flatnonzero(record_high_mask.to_numpy())
len(record_high_positions)

1510

In [10]:
correction_records = []
interval_ends = list(record_high_positions[1:]) + [len(sp500_close)]

for peak_position, next_high_position in zip(record_high_positions, interval_ends):
    interval = sp500_close.iloc[peak_position:next_high_position]
    peak_date = sp500_close.index[peak_position]
    peak_price = sp500_close.iloc[peak_position]
    trough_date = interval.idxmin()
    trough_price = interval.min()
    drawdown_pct = (peak_price - trough_price) / peak_price * 100

    if drawdown_pct >= 5:
        correction_records.append(
            {
                "peak_date": peak_date,
                "trough_date": trough_date,
                "peak_close": peak_price,
                "trough_close": trough_price,
                "drawdown_pct": drawdown_pct,
                "duration_days": (trough_date - peak_date).days,
            }
        )

corrections = pd.DataFrame(correction_records).sort_values("peak_date").reset_index(drop=True)
corrections.tail()

,peak_date,trough_date,peak_close,trough_close,drawdown_pct,duration_days
69,2024-03-28,2024-04-19,"5,254.3501","4,967.2300",5.4644,22
70,2024-07-16,2024-08-05,"5,667.2002","5,186.3301",8.4851,20
71,2025-02-19,2025-04-08,"6,144.1499","4,982.7700",18.9022,48
72,2025-10-28,2025-11-20,"6,890.8901","6,538.7598",5.1101,23
73,2026-01-27,2026-03-30,"6,978.6001","6,343.7202",9.0975,62


In [11]:
correction_percentiles = corrections[["drawdown_pct", "duration_days"]].quantile([0.25, 0.50, 0.75])
largest_corrections = corrections.nlargest(10, "drawdown_pct")

display(correction_percentiles)
display(largest_corrections)
print(f"Answer: the median significant drawdown is {correction_percentiles.loc[0.50, 'drawdown_pct']:.2f}%.")
print(f"The median peak-to-trough duration is {correction_percentiles.loc[0.50, 'duration_days']:.0f} calendar days.")

,drawdown_pct,duration_days
0.2500,6.2347,22.0000
0.5000,7.9864,40.5000
0.7500,14.0198,86.2500


,drawdown_pct,duration_days
0.2500,6.2347,22.0000
0.5000,7.9864,40.5000
0.7500,14.0198,86.2500


,peak_date,trough_date,peak_close,trough_close,drawdown_pct,duration_days
56,2007-10-09,2009-03-09,"1,565.1500",676.5300,56.7754,517
54,2000-03-24,2002-10-09,"1,527.4600",776.7600,49.1469,929
24,1973-01-11,1974-10-03,120.2400,62.2800,48.2036,630
22,1968-11-29,1970-05-26,108.3700,69.2900,36.0616,543
65,2020-02-19,2020-03-23,"3,386.1499","2,237.3999",33.9250,33
35,1987-08-25,1987-12-04,336.7700,223.9200,33.5095,101
15,1961-12-12,1962-06-26,72.6400,52.3200,27.9736,196
27,1980-11-28,1982-08-12,140.5200,102.4200,27.1136,622
68,2022-01-03,2022-10-12,"4,796.5601","3,577.0300",25.4251,282
18,1966-02-09,1966-10-07,94.0600,73.2000,22.1773,240


,drawdown_pct,duration_days
0.2500,6.2347,22.0000
0.5000,7.9864,40.5000
0.7500,14.0198,86.2500


,peak_date,trough_date,peak_close,trough_close,drawdown_pct,duration_days
56,2007-10-09,2009-03-09,"1,565.1500",676.5300,56.7754,517
54,2000-03-24,2002-10-09,"1,527.4600",776.7600,49.1469,929
24,1973-01-11,1974-10-03,120.2400,62.2800,48.2036,630
22,1968-11-29,1970-05-26,108.3700,69.2900,36.0616,543
65,2020-02-19,2020-03-23,"3,386.1499","2,237.3999",33.9250,33
35,1987-08-25,1987-12-04,336.7700,223.9200,33.5095,101
15,1961-12-12,1962-06-26,72.6400,52.3200,27.9736,196
27,1980-11-28,1982-08-12,140.5200,102.4200,27.1136,622
68,2022-01-03,2022-10-12,"4,796.5601","3,577.0300",25.4251,282
18,1966-02-09,1966-10-07,94.0600,73.2000,22.1773,240


Answer: the median significant drawdown is 7.99%.
The median peak-to-trough duration is 40 calendar days.


## Question 4: Amazon earnings surprises

For an earnings date on Day 2, the two-day return is `Close_Day3 / Close_Day1 - 1`. Dates are normalized and timezone information is removed before joining earnings to prices.

In [12]:
amazon = yf.Ticker("AMZN")
earnings = amazon.get_earnings_dates(limit=30).copy()
earnings.index = pd.to_datetime(earnings.index).tz_localize(None).normalize()
earnings = earnings.loc[~earnings["Reported EPS"].isna()].copy()
earnings.tail()

,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2015-07-23,-0.0100,0.0100,240.3200
2015-04-23,-0.0100,-0.0100,-12.1500
2015-01-29,0.0100,0.0200,152.5300
2014-10-23,-0.0400,-0.0500,-23.7300
2014-07-24,-0.0100,-0.0100,-83.4200


In [13]:
amazon_prices = amazon.history(period="max", auto_adjust=False)[["Close"]].copy()
amazon_prices.index = pd.to_datetime(amazon_prices.index).tz_localize(None).normalize()
amazon_prices["two_day_return"] = (
    amazon_prices["Close"].shift(-1) / amazon_prices["Close"].shift(1) - 1
)

earnings_analysis = earnings.join(amazon_prices[["two_day_return"]], how="left")
positive_surprises = earnings_analysis.loc[earnings_analysis["Surprise(%)"] > 0].dropna(
    subset=["two_day_return"]
)
positive_surprises[["Reported EPS", "Surprise(%)", "two_day_return"]].sort_index()

,Reported EPS,Surprise(%),two_day_return
Earnings Date,,,
2015-01-29,0.0200,152.5300,0.1666
2015-07-23,0.0100,240.3200,0.0843
2015-10-22,0.0100,231.1700,0.0778
2016-04-28,0.0500,76.9200,0.0874
2016-07-28,0.0900,63.7200,0.0301
2017-02-02,0.0800,9.7000,-0.0266
2017-04-27,0.0700,34.3000,0.0173
2017-10-26,0.0300,"3,900.0000",0.1316
2018-02-01,0.1100,16.4300,-0.0144


In [14]:
median_positive_surprise_return = positive_surprises["two_day_return"].median()
surprise_return_correlation = positive_surprises[["Surprise(%)", "two_day_return"]].corr().iloc[0, 1]

print(f"Answer: median two-day return after a positive surprise = {median_positive_surprise_return:.2%}.")
print(f"Correlation between surprise magnitude and two-day return = {surprise_return_correlation:.4f}.")

Answer: median two-day return after a positive surprise = 2.32%.
Correlation between surprise magnitude and two-day return = 0.3290.


## Question 5: Capstone idea (optional)

**Draft answer:**

I want to build a **Multi-Chain Micro-Cap Crypto Quantitative Backtesting & Strategy Comparison Engine** to systematically identify the most profitable intraday trading strategy for pump-and-dump and breakout dynamics in decentralized exchange (DEX) tokens.

- **Asset universe:** Newly launched and trending micro-cap tokens across major DEX ecosystems (Solana Raydium/Pump.fun, Base Uniswap/Aerodrome, Ethereum Uniswap v2/v3, and BNB Chain PancakeSwap) with initial liquidity between $10k and $1M.
- **Core Research Question / Strategy Comparison:**
  - Build a unified event-driven backtesting engine to compare 3 distinct strategy archetypes across historical pool lifecycles:
    1. **Early Momentum Breakout (Riding the Pump):** Entering on rapid buyer volume surges within 5–30 minutes of pool creation with dynamic trailing stops.
    2. **Bonding-Curve / Migration Momentum:** Trading liquidity migration events (e.g., tokens graduating from bonding curves to full DEX pools).
    3. **Post-Peak Exhaustion Fading (Shorting / Fading the Dump):** Detecting momentum divergence and liquidity depletion to short or fade late-stage pump cycles on DEX perpetual markets.
- **Prediction target and horizon:**
  - **Target:** Multi-class / Triple-Barrier labeling (Success: $+50\%$ to $+100\%$ upside before $-20\%$ drawdown; Failure / Rug: $-50\%$ to $-100\%$; Neutral: flat threshold).
  - **Horizon:** Intraday timeframe (5 minutes to 4 hours holding window).
- **Candidate features:**
  - **On-Chain Microstructure:** Buyer-to-seller wallet ratio, Top-10 holder supply concentration %, Dev wallet token retention / sell actions, LP burn/lock status, and liquidity-to-market-cap ratio.
  - **Flow & Order Velocity:** 1-min and 5-min Cumulative Volume Delta (CVD), volume-to-liquidity acceleration ($V/L$ ratio $>3\times$), and transaction count velocity ($\Delta \text{tx} / \text{min}$).
  - **Safety / Rug Filtering:** Contract security flags (mint authority status, blacklist/honeypot tests via RugCheck / GoPlus APIs).
- **Model and baseline:**
  - **Models:** LightGBM / CatBoost classifier with probability threshold calibration to maximize precision (prioritizing false positive / rug avoidance) combined with an adaptive sizing rule.
  - **Baselines:** 1) Naive volume spike breakout rule, 2) Equal-weight buy-and-hold (benchmarks survivorship loss), 3) Fixed take-profit/stop-loss heuristic.
- **Evaluation and backtest engine:**
  - **Realistic Friction Modeling:** Constant-product AMM slippage ($x \cdot y = k$) based on simulated pool reserves, variable DEX swap fees ($0.25\% - 1\%$), and priority transaction / gas fee deductions per trade.
  - **Performance Metrics:** Net Profit Factor, Sharpe Ratio, Expectancy per trade after fees, Maximum Drawdown, and Rugpull Avoidance Rate (% of malicious tokens filtered before entry).
- **Main risks or limitations:**
  - MEV sandwich attacks and adverse execution front-running.
  - Survivorship and selection bias in historical DEX data (mitigated by pulling full initial block/pool creation logs including dead tokens).

## Question 6: New metrics (optional)

**Draft answer:**

| Metric | Source / retrieval method | Why it may be useful |
|---|---|---|
| **DEX Volume-to-Liquidity Ratio ($V/L$)** | GeckoTerminal API / DexScreener API (`/networks/{network}/pools/{pool}`) | Identifies sudden volume surges relative to pool depth ($V/L > 3.0$), signaling early breakout momentum before price spikes. |
| **Unique Buyer vs. Seller Wallet Ratio ($\Delta \text{Wallets}$)** | Birdeye API / DEX swap event logs via RPC or BigQuery crypto datasets | Distinguishes organic community buying from single-wallet wash trading / developer manipulation. |
| **Liquidity Pool Lock / Burn Percentage** | RugCheck API / GoPlus Security API (`/token_security`) | Critical safety filter to screen out high-probability honeypots and rugpulls where developers retain the ability to pull pool liquidity. |
| **Top 10 Holder Concentration %** | On-chain token supply distribution via Solscan / Etherscan API | Quantifies dump risk: high concentration ($>40\%$ in non-LP wallets) indicates high vulnerability to insider dump cascades. |

## Submission checklist

- Run all cells from a clean kernel.
- Check that the downloaded date ranges match the homework cutoff.
- Record the printed answers in the submission form.
- Complete the optional written answers if desired.

Submission form: https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw01